# [LOG04 미션 08] AAR 회의를 통한 비효율적 물자 할당 사례 분석 및 우선순위 기준 보정 — 학생 실습용 (Student)

- **산출물 경로**: `reports/log04_redteam_critique_report.md`
- **실습 환경**: Python 3.12 표준 라이브러리 (`pathlib`, `csv`) — 별도 라이브러리 설치 및 인터넷 연결 불필요

> **실습·보안 안내**: 제공된 교육용 가상자료만 사용하며 실제 조직명·물자 제원·재고·개인정보는 입력하지 마세요. 점수나 등급으로 승인을 자동 결정하지 않으며, 최종 판단은 반드시 사람이 내립니다.

---

## 미션 8. AAR 회의를 통한 비효율적 물자 할당 사례 분석 및 우선순위 기준 보정

### 목표

이번 미션에서는 교육장에서 제공하는 생성형 AI와 함께 「물자 배분 사례의 원인 분석과 수정 계획」 초안을 만듭니다.

과잉 배분과 부족은 서로 다른 결과지만 기록이 늦거나 확인 순서가 빠진 문제에서 함께 생길 수 있습니다.

사람을 탓하기보다 어떤 문장을 고칠지 찾는 문서를 씁니다.

준비할 자료는 다음과 같습니다: 업무 일지 A·B, 미션 2·4·5 기준. AI가 제안한 초안을 받은 뒤 사실과 숫자, 빠진 항목, 과장된 표현을 확인하고 직접 고쳐 씁니다.

끝나면 완성한 문서와 자료 대조 기록, 사용한 프롬프트, 고친 이유가 남습니다.

문장마다 근거를 남겨 다음 미션에서도 같은 자료를 찾아 쓸 수 있게 합니다.

수량은 단위와 기간을 함께 적고, 계산 결과는 계산기로 다시 확인합니다.

자료에 없는 값은 추측하지 않고 미확인으로 남깁니다.

다른 담당자가 문장과 표를 읽고 같은 자료를 찾을 수 있게 근거를 붙입니다.

### 상황과 자료

가상 업무 일지 A·B에는 과잉 배정과 자료 지연 사례가 남았습니다.

A팀은 기준서의 전제를 검토하고 B팀이 납득할 수정 이유를 적으려 합니다.

**준비할 자료**: 업무 일지 A·B, 미션 2·4·5 기준

**뜻 알아두기**: AAR은 일을 마친 뒤 계획과 실제를 비교하는 돌아보기입니다.

레드팀은 기준의 허점을 찾는 검토자입니다.

전부 채택하거나 모두 보류해도 각 근거와 이유를 남깁니다.

이 미션은 선택입니다.


In [ ]:
# [환경 준비] 미션 8 실습 데이터셋 및 빈 산출물 서식 생성 (기존 파일은 보존)
from pathlib import Path

payload = {'02_데이터셋/거점_보유소모_대장.csv': "[교육용가상자료] 거점 보유·소모 대장 — 가상 조직 A팀 3개 거점의 D-3일~D일 기록. 수량 단위는 품목별(발·세트·묶음). 실제 재고·제원과 무관하다. 첫 줄은 안내문, 둘째 줄부터 머리글.\n거점,품목,단위,D-3일보유량,D-2일소모,D-1일소모,D일소모,입고량_D-2~D,D일보유량_보고,일평균소모량_보고,리드타임_일,보고출처,보고시각,비고\n북곡,AM-1,발,42,6,6,6,0,24,6,3,창고 실사+서명,D일 17:00,D-3일 소모는 3발(이전 수준)\n북곡,DP-2,세트,60,5,5,5,0,40,5,2,불출 대장 집계,D일 17:00,파손 손실 기록란 비어 있음\n북곡,PG-3,묶음,150,10,10,10,0,120,10,1,창고 실사,D일 17:00,\n서림,AM-1,발,27,3,3,3,0,18,3,3,담당자 A 대장 집계,D일 18:00,담당자 B는 소모 5/일로 보고(K4)\n서림,DP-2,세트,24,6,6,0,0,12,4,2,불출 대장 집계,D일 18:00,D일 소모는 18시 집계 전(0으로 기재). 파손 손실과 정상 소모가 한 칸에 합산\n서림,PG-3,묶음,75,15,15,15,0,30,15,1,담당자 구두,D일 18:00,수량 '30 정도'의 어림값\n해솔,AM-1,발,20,4,4,4,0,8,4,3,전일 보고서 인용,D-1일 18:00,전일 인용은 D-1일 18시 출처 시각이다. D일 19시 대장 담당자가 D일 소모 4를 계산용 추정 기입하고 보유량 8을 산출했다. D일 실사값 아님. 현재량 확인 후 사용\n해솔,DP-2,세트,26,2,2,2,0,20,2,2,창고 실사,D일 16:00,\n해솔,PG-3,묶음,78,6,6,6,0,60,6,1,창고 실사,D일 16:00,야간 소모량 미반영\n", '02_데이터셋/재보급_요청대장.csv': "[교육용가상자료] 재보급 요청대장 — D일 가상 조직 A팀이 지원팀에 올린 요청. '요청서기재_보유일수'는 요청 거점이 스스로 적은 값이며 대장 재계산과 다를 수 있다. 첫 줄은 안내문, 둘째 줄부터 머리글.\n요청ID,거점,품목,요청수량,요청시각,요청사유,요청서기재_보유일수,근거유형,수송로상태,도착예정,비고\nQ4,해솔,AM-1,10,D일 07:30,보유일수 2일로 리드타임 3일보다 짧음,2.0,전일 보고서 인용,양호,D+3일,R3와 같은 자료\nQ1,북곡,AM-1,12,D일 09:00,현장 대응 빈도 증가로 소모 6발/일,4.0,창고 실사+서명,양호,D+3일,R1과 같은 자료\nQ3,해솔,PG-3,60,D일 11:00,예방,4.0,요청서 1장(실사·대장 없음),양호,D+1일,C3와 같은 요청. 사유가 '예방'뿐\nQ5,서림,PG-3,30,D일 14:00,일 소모 15묶음,2.0,담당자 구두,양호,D+1일,R4와 같은 자료\nQ6,북곡,DP-2,20,D일 16:00,예방,8.0,근거 없음,양호,D+2일,B4와 같은 요청\nQ2,서림,DP-2,16,D일 18:30,이틀 만에 절반 소진(파손 포함 여부 불명),3.0,불출 대장 집계,지연 1일,D+3일,R2·C2와 같은 자료\n", '02_데이터셋/20261019b_선택미션_대체입력.md': '# [교육용가상자료] 선택 미션 미수행 시 제공 대체입력\n\n필수 미션 1·미션 2·미션 3·미션 5·미션 7·미션 9·미션 10의 Q1·Q2·Q3 모두 PASS일 때 최종 PASS다. 선택 미션 4·미션 6·미션 8은 산출물과 Q1·Q2·Q3 모두 가점/참고이며 최종 PASS에서 제외합니다. 선택 미션 미수행 시 02_데이터셋/20261019b_선택미션_대체입력.md의 제공 기본값·빈 서식을 사용한다. 후속 필수 미션에서 그 입력을 적용한 증거는 평가하지만 선택 미션 수행물·AI 세션을 소급 요구하지 않는다.\n\n제공 기본값은 정답도, 학습자 수행물도 아니다. 후속 미션에서 사용한 버전·값·자료 근거를 표시한다.\n\n## 미션 4 대체 규칙 v1\n\n제공 근거 가중치: 보유일수<리드타임 +3, 직전 2일보다 당일 소모 증가 +2, G4·G5·G6 -3, 근거 없는 예방 요청 -2. 모르는 값은?로 별도 기록한다. 종합 점수 = 상향 근거 가중치의 합 − 하향 근거 가중치 절댓값의 합. 규칙1: 점수3 이상은 긴급 승인 검토 후보. 규칙2: G4 이상 또는 독립 교차확인 불일치·미확인이면 보류. 규칙3: 확인 불가 1개 이상이면 추가 확인 요청. 규칙4: 사람의 최종 결정·운영 규칙 우선. 규칙2·3 보류 조건은 점수보다 우선한다. 미션 4의 12항목 설계·S1~S3·레드팀 제출은 요구하지 않는다.\n\n## 미션 6 대체 빈 서식\n\n| 요청ID | 거점 | 품목 | 보유일수 | 여유 | 위험 등급 | 요청수량 | 판단·근거 |\n|---|---|---|---|---|---|---|---|\n| Q1 | | | | | | | |\n| Q2 | | | | | | | |\n| Q3 | | | | | | | |\n| Q4 | | | | | | | |\n| Q5 | | | | | | | |\n| Q6 | | | | | | | |\n\n미션 9에서 필요한 현황은 미션 1·미션 3 결과로 이 빈 서식을 채운다. 미션 6의 메모3종 변환·금지표현6개는 요구하지 않는다.\n\n## 미션 8 대체 기준 유지안 v1\n\n이전 기준=미션 2·미션 3·미션 5 보존본, 수정 기준=동일 기준 유지안 v1. 미션 9의 V1~V7에 각각 재적용하고 동일 결과는 근거·유지 이유로 기록한다. 제공된 과거 판단과 내 이전 기준은 분리한다. 미션 8 쟁점5건·반박·AI 세션 제출은 요구하지 않는다.\n\n## 미션 10 부록\n\n수행한 선택 미션은 실제 산출물을, 미수행한 선택 미션은 이 제공 기본값·빈 서식과 후속 필수 미션에서 작성한 적용 기록을 부록에 넣는다. 원본은 보존하고 기준 유지·변경 내역을 표시한다.\n\n## 사실·판단·조치·미확인 분류\n\n사실은 제공 자료에서 확인한 내용, 판단은 사실을 기준에 적용한 해석, 조치는 다음에 할 행동, 미확인은 근거가 부족한 항목입니다.\n\n| 항목 | 사실과 자료 ID | 판단과 적용 기준 | 조치 | 미확인·재확인 대상 |\n|---|---|---|---|---|\n| 직접 작성 | | | | |\n', '03_참고자료/20261019b_정오표.md': '# 실습 적용 기준 — 원문 보존 / 수정본 채점\n\n원문은 보존 원문 및 학습자용 보존본에 남아 있습니다. 원문 대조용 보존본은 유지합니다. 현재 실습에는 아래 표의 수정 값을 적용하세요.\n\n| 항목 | 원문 값/종전 값 | 수정 값 | 이유 | 채점 적용 범위 |\n|---|---|---|---|---|\n| 선택 미션 평가 | 선택 미션 Q1만 선택, 산출물·Q2·Q3 필수(이전 안내) | 미션 4·미션 6·미션 8 산출물·Q1·Q2·Q3 모두 가점/참고 | 최종 PASS 범위 충돌 해소 | 필수 7미션의 Q1·Q2·Q3만 최종 PASS |\n| G 척도 경계 | G 이하/이상 혼용; 규칙2=미션 2 값 | 보류G≥4 / 통과후보G≤3 및 독립 교차확인 일치 | 경계 G3 중첩 제거 | 미션 2 서식4·미션 4 규칙2·미션 5 분기2 |\n| 전문 검토 순서 | 분기1→2→3→4→5 | 번호 유지, 순서4→3→1→2→5 | 전문 검토를 위험·신뢰도보다 먼저 의뢰 | B3 기술검토 분기3, B5 법무검토 분기4; 승인 아님 |\n| 분할 입력 | 길이 제한 안내: 시작 문서7절 | 현재 수행 단계 안의 분할 입력 절차 | 없는 참조 제거 | 미션 2 AI 비교 |\n| 보고 시각·추정값 | 전일 보고 인용인데 D일 소모4·보유8 | 전일 출처 시각 + D일19시 계산용 추정기입 명시 | 실사 시각과 추정 계산 구분; 수치 불변 | 해솔AM-1은 현재량 확인 후 사용, G4 유지 |\n| 요청 수신처·요약행 | M-1 본부 요청 / 요약표4행 | Q2 지원팀 요청(후속상신 미확인) / Q1~Q6 6행 | 수신처·행 수 정합 | 미션 6 실습 자료 서식1 |\n| 치환 기록 중복 | 치환 자료 끝 문단2회 | 동일 지시1회 | 중복 제거 | 미션 10 치환 적용 기록 |\n| 도메인 예시 | .go.kr 예시에 교육장 제공 자료 | .or.kr 예시로 이동; 기관·발행자 확인 우선 | 도메인 형태와 신뢰도 혼동 방지 | 학습 안내 문서; 접속 확인 아님 |\n', '03_참고자료/20261019_보완정오표.md': '# 2026-10-19 보완 실습 적용 기준\n\n원문 계열 표·목표를 덮어쓰지 않고 별도 보완 실습본을 제공합니다. 아래 안내에 따라 대장과 용어를 해석하세요.\n\n| 항목 | 원문 값/종전 안내 | 수정 값/보완 안내 | 이유 | 채점 적용 범위 |\n|---|---|---|---|---|\n| 미션 9 재검증 | 원문 서식 2의 개선 전 판단은 제공표 값 | 원문 값 유지 + 별도 표에 내 이전 기준 재적용 판단·수정 기준 적용 판단 추가 | 과거 판단과 내 기준 개정 효과의 혼동 방지 | 미션 9 재검증 7건의 과정·근거; 원문 Q1 문장·제공표·미측정 과거 시간 불변 |\n| 미션 7 카드 범위 | 원문 목표·수행은 거점 교대 포함, T1~T6에는 기준선 포함·거점 교대 없음 | 원문 불변, 거점 교대는 제공 카드 학습 범위 밖이라고 명시 | 실제 카드로 수행한 범위만 보고 | 제공 T1~T6만 수행·채점; 추가 카드·평가 조건 없음 |\n| 미션 1 후보 검토 | AI 후보 전부 채택이면 보완 | 전체 후보의 근거 대조·유지 이유 없이 전부 채택하면 보완 | 폐기 0건 허용 원칙과 정합화 | 미션 1 수행 절차의 통과 기준·Q2와 파생 문서; 근거 있는 전부 채택 인정 |\n| 멈춘 분기의 뜻 | 아니오가 나와 보류로 빠지는 분기 번호 | 예/아니오와 관계없이 보류·추가 확인·전문 검토로 넘긴 지점의 분기 번호 | 예 경로 중단도 기록 가능해야 함 | 시작 용어표·미션 5·미션 10 기록 안내, 원문 트리 불변 |\n', '02_데이터셋/log04_m08_sample.md': "# LOG04 미션 8 실습 샘플\n\n> `02_데이터셋/LOG04_실습자료_수정실습본.md`의 미션 8 부분입니다. 전체 자료는 프로젝트 첨부 자료의 학습자배포 ZIP에 있습니다.\n\n**미션 8. AAR 회의를 통한 비효율적 물자 할당 사례 분석 및 우선순위 기준 보정**\n\n가. 제공 자료 - 가상 AAR 일지 A·B [교육용가상자료]\n\n| **시각** | **AAR 일지 A (과잉 배정 사례)** | **AAR 일지 B (자료 지연 부족 사례)** |\n| --- | --- | --- |\n| **D+1 08:00** | 해솔 PG-3 추가 배정 요청 접수(보유일수 10일) | 서림 DP-2 보고 도착, 전일 18시 자료 |\n| **D+1 10:30** | 요청 사유 '예방', 신뢰도 확인 없이 승인 | 지원팀 비축 확인, 서림 요청은 일반 보급으로 분류 |\n| **D+1 15:00** | 해솔에 PG-3 60묶음 추가 도착 | 서림 DP-2 소모 급증, 보유 4세트로 감소 |\n| **D+2 07:00** | 서림 PG-3 보유 6묶음, 긴급 요청 발생 | 서림 긴급 요청, 리드타임 2일 |\n| **D+2 12:00** | 북곡에서 서림으로 물자 이동 검토, 수송 불가 | D+3 도착 예정, 그동안 서림 DP-2 보유 0세트 |\n\n나. 작성 서식 1 - 계획과 실제 차이, 원인 분류(사람·절차·정보·환경, 개인 비난 표현 금지)\n\n| **번호** | **계획** | **실제** | **원인 분류** | **원인 설명(사실 중심)** | **개선안** |\n| --- | --- | --- | --- | --- | --- |\n| **1** | | | | | |\n| **2** | | | | | |\n| **3** | | | | | |\n| **4** | | | | | |\n| **5** | | | | | |\n\n나. 작성 서식 2 - 레드팀 쟁점·대응(5건 이상)\n\n| **번호** | **기준서의 취약한 전제** | **레드팀 비판** | **작성 팀 반박 또는 수정안** | **경고 문구** |\n| --- | --- | --- | --- | --- |\n| **1** | | | | |\n| **2** | | | | |\n| **3** | | | | |\n| **4** | | | | |\n| **5** | | | | |\n| **6** | | | | |\n\n나. 작성 서식 3 - 반대 방향 위험 점검\n\n| **위험** | **설명** | **대응** |\n| --- | --- | --- |\n| **한 거점에 몰아주기 위험** | | |\n| **예비 물자 소진 위험** | | |\n| **기준 복잡도 증가로 현장 판단 지연** | | |\n\n나. 작성 서식 4 - 수정 반영 계획\n\n| **번호** | **수정할 문서와 위치** | **수정 내용** | **책임 직책(가상)** | **기한** |\n| --- | --- | --- | --- | --- |\n| **1** | | | | |\n| **2** | | | | |\n| **3** | | | | |\n| **4** | | | | |\n\n다. 점검표\n\n| **점검 항목** | **확인** |\n| --- | --- |\n| 일지 A·B의 계획과 실제 차이를 5건 이상 적었다 | ( ) |\n| 원인을 사람·절차·정보·환경으로 분류하고 개인 비난 표현이 없다 | ( ) |\n| 레드팀 쟁점 5건 이상에 반박 또는 수정안을 100% 적었다 | ( ) |\n| 취약한 전제에 경고 문구를 명시했다 | ( ) |\n| 반대 방향 위험 3가지를 점검했다 | ( ) |\n", 'reports/log04_redteam_critique_report.md': '# 미션 8 reports/log04_redteam_critique_report.md(AAR 원인 분석표, 레드팀 쟁점·대응표, 우선순위 기준 보정 계획) (선택·가점/참고)\n\n빈 작성 양식입니다. 실제 수행 후 작성하세요.\n\n## 산출물 본문\n\n(여기에 작성)\n\n## 자료 ID와 판단 근거\n\n(여기에 작성)\n\n## 협업·AI 기록과 선별 이유\n\n(여기에 작성)\n\n## 점검 결과와 미수행 항목\n\n(여기에 작성)\n'}
for name, content in payload.items():
    path = Path(name)
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        path.write_text(content, encoding='utf-8')
print(f'미션 8 자료 준비 완료 ({len(payload)}개 파일 확인, 기존 작성 파일 보존)')


### 할 일

1. 일지별 시간순 기록에서 계획과 실제의 차이 5건 이상을 찾습니다.

2. AI에게 사람·절차·정보·환경으로 원인을 나눈 보고서 초안을 요청합니다.

3. 새 대화에서 취약한 전제 5건 이상을 비판받고 자료와 대조합니다.

4. 반대 방향 위험 3종과 수정할 조항을 연결합니다.

5. 원인표·쟁점 대응표·수정 계획·3열 인과표를 저장합니다.

평가의 자료 수·숫자·근거 조건은 같은 미션 문제지의 확인 포인트와 대조합니다.

**핵심 수행·평가 포인트 (선택·가점/참고)**
- 선택 미션이며 미수행 시 `02_데이터셋/20261019b_선택미션_대체입력.md`의 기준 유지안을 사용합니다.
- 서식 1(5행 이상)은 일지 A·B의 시각·수량(10일, 60묶음, 6묶음, 4세트, 0세트)을 원문 그대로 유지하고 일지 A를 Q3 불일치와 연결하며, 원인 문장에 사람 이름·`'부주의·태만·실수'` 등 비난 표현을 0건으로 유지합니다.
- 별도 새 대화에서 레드팀 반박 5건 이상을 받아 수용/기각 이유를 적고, 서식 4(4행 이상)는 미션 2·3·5 조항 번호와 전·후 문구로 적습니다.
- 3열 인과표 5행의 셋째 열에는 미션 9 재검증 V1~V7 사례와 연결해 기록합니다.


### AI와 함께 해보기

교육장에서 제공하는 생성형 AI에 아래 요청을 복사하고 자료를 채웁니다.

```text
너는 가상 조직의 문서 작성 도우미야.
아래 자료만 사용해 「물자 배분 사례의 원인 분석과 수정 계획」의 초안을 만들어 줘.
자료: 업무 일지 A·B, 미션 2·4·5 기준.
[여기에 해당 가상 자료와 내 기준을 붙여넣기]
형식: 시각 | 계획 | 실제 | 차이 | 근거. 원인 | 조항 | 수정안 | 반대 위험. 지적 | 바꾼 줄 | 달라진 결과.
조건: 자료 ID를 붙여 줘. 사실·숫자·시각을 만들지 말고 알 수 없으면 미확인으로 써 줘. 단정 대신 근거와 한계를 함께 적어 줘. 최종 판단은 내가 할게.
```

**확인할 것 3가지**

1. 사실·숫자: 자료 ID, 원문, 수량, 단위, 시각을 자료와 대조합니다.

2. 빠진 항목: 요청한 표의 열과 미확인·근거·책임 항목이 있는지 봅니다.

3. 과장 표현: 근거 없이 반드시·확실하다라고 쓴 문장을 찾습니다.

### 작성 서식

더블클릭해 빈칸을 채우고 `Shift+Enter`로 표시하세요. 별도 산출물 파일에도 옮겨 저장할 수 있습니다.

**미션 8. 물자 배분 사례의 원인 분석과 수정 계획 (선택·가점/참고)**

나. 작성 서식 1 - 계획과 실제 차이, 원인 분류(사람·절차·정보·환경, 개인 비난 표현 금지)

| **번호** | **계획** | **실제** | **원인 분류** | **원인 설명(사실 중심)** | **개선안** |
| --- | --- | --- | --- | --- | --- |
| **1** | | | | | |
| **2** | | | | | |
| **3** | | | | | |
| **4** | | | | | |
| **5** | | | | | |

나. 작성 서식 2 - 레드팀 쟁점·대응(5건 이상)

| **번호** | **기준서의 취약한 전제** | **레드팀 비판** | **작성 팀 반박 또는 수정안** | **경고 문구** |
| --- | --- | --- | --- | --- |
| **1** | | | | |
| **2** | | | | |
| **3** | | | | |
| **4** | | | | |
| **5** | | | | |
| **6** | | | | |

나. 작성 서식 3 - 반대 방향 위험 점검

| **위험** | **설명** | **대응** |
| --- | --- | --- |
| **한 거점에 몰아주기 위험** | | |
| **예비 물자 소진 위험** | | |
| **기준 복잡도 증가로 현장 판단 지연** | | |

나. 작성 서식 4 - 수정 반영 계획

| **번호** | **수정할 문서와 위치** | **수정 내용** | **책임 직책(가상)** | **기한** |
| --- | --- | --- | --- | --- |
| **1** | | | | |
| **2** | | | | |
| **3** | | | | |
| **4** | | | | |

다. 점검표

| **점검 항목** | **확인** |
| --- | --- |
| 일지 A·B의 계획과 실제 차이를 5건 이상 적었다 | ( ) |
| 원인을 사람·절차·정보·환경으로 분류하고 개인 비난 표현이 없다 | ( ) |
| 레드팀 쟁점 5건 이상에 반박 또는 수정안을 100% 적었다 | ( ) |
| 취약한 전제에 경고 문구를 명시했다 | ( ) |
| 반대 방향 위험 3가지를 점검했다 | ( ) |

### AI 대조·수정 기록 및 남길 기록

| 자료 ID | 초안 문장 | 고친 문장 | 고친 이유 |
|---|---|---|---|
| | | | |

**남길 기록**

1. 프롬프트 전문과 실제 응답(연속 5줄 이상, 짧으면 전문), 세션 구분(협업 시 역할·요청·독립 판단 원문)

2. 사실·숫자 대조 / 빠진 항목 / 과장 표현 확인

3. 전체 후보 대조 및 채택·폐기(유지) 이유

4. 점검 결과와 미수행 항목


In [ ]:
# [학생 보조 도구] 미션 8 샘플·산출물 파일 확인
from pathlib import Path

sample = Path("02_데이터셋/log04_m08_sample.md")
out = Path("reports/log04_redteam_critique_report.md")
print(f"샘플 파일 존재: {sample.exists()} ({len(sample.read_text(encoding='utf-8'))}자)")
status = "미작성(빈 템플릿)" if "빈 작성 양식입니다." in out.read_text(encoding="utf-8") else "작성됨"
print(f"산출물 파일({out}): {status}")
